In [1]:
import pdfplumber
import pandas as pd
import numpy as np

In [2]:
import re

In [3]:
from collections import Counter

In [4]:
path = 'D:/Downloads/shivamQ2.pdf'

In [5]:
file1 = 'D:/Downloads/shivamBefore.pdf'

In [6]:
def duplicate_handler(series):
    things = Counter(series)
    for key, value in things.items():
        if value > 1:
            toHandle = key

    return series[series == toHandle].index.tolist()

In [7]:
s = pd.Series(['Alice', 'Bob', 'Alice', 'Charlie'])

print(s[s == 'Alice'].index)
print(duplicate_handler(s))

Index([0, 2], dtype='int64')
[0, 2]


In [87]:
def financial_statements(filePath):

    finalThings = list()
    
    with pdfplumber.open(filePath) as pdf:
        pages = pdf.pages
        wholePage = pages[0].extract_text()
        statements = wholePage.split('udited Statement of ')
        
        for statement in statements:
            dataList = []
            lines = statement.split('\n')
            metricName = None
            headersNotFound = True
            sectionName = None
            #print(statement)
            thCounter = 0
            for ind, line in enumerate(lines):
                if ind == 0:
                    title = line
                    print(title)
                if headersNotFound:
                    if line.strip() == "Particulars":
                        line = line + ' ' + lines[ind + 1]
                    headingCheck = re.findall(r'^(\d\d)[\D]+\s', line)
                    if ('Particulars' in line and line.strip() != 'Particulars' and not metricName) or headingCheck:
                        words = line.split(' ')
                        metricName = words.pop(0)
                        print(words)
                        for word in words:
                            print(word)
                            if 'th ' in word or 'rd ' in word:
                                thCounter += 1
                        if thCounter == 0:
                            thCounter = 2
                        print('==============thCounter=============', thCounter)
                        joiningIndex = int(len(words) / thCounter)
                        firstDate = ' '.join(words[:joiningIndex])
                        secondDate = ' '.join(words[joiningIndex:])
                        headersNotFound = False
                else:
                    print(line)
                    rePattern = r'^([\D ]+)\s+(\(?[\d,\)]+\)?|-)?\s+(\(?[\d,]+\)?|-)?$'
                    rePattern2 = r'^([\D ]+)\s+(\(?[\d,\)]+\)?|-)?\s+(\(?[\d,]+\)?|-)?\s+(\(?[d,]+\)?)$'
                    regexPattern = re.findall(rePattern, line)
                    if ind != len(lines) - 1:
                        if ind > 0 and 'Total' in lines[ind - 1] and 'Total' not in line and 'Financial Position' in title:
                            sectionName = line.strip(': ')
                        elif not regexPattern and re.findall(rePattern, lines[ind+1]) and len(line.split(' ')) < 5:
                            sectionName = line.strip(': ')
                        #elif not regexPattern and len(line.split(' ')) < 4:
                            #sectionName = line.strip(': ')
                    if not regexPattern:
                        regexPattern = re.findall(rePattern2, line)
                    if regexPattern:
                        valuesTemp = regexPattern[0]
                        print(regexPattern)
                        dataList.append({'section': sectionName,
                                        metricName: valuesTemp[0],
                                        firstDate: valuesTemp[1],
                                        secondDate: valuesTemp[2]})
            if dataList:
                dataFrame = pd.DataFrame(dataList)
                duplicates = dataFrame['Particulars'].duplicated(keep=False)
                trueSeries = dataFrame['section'] + '_' + dataFrame['Particulars']
                newOnes = pd.Series(np.where(duplicates, trueSeries, dataFrame['Particulars']))
                newDuplicates = newOnes.duplicated()
                #print(f'new things: {newOnes}')
                repeatedGroups = newOnes.groupby(newOnes).cumcount()
                numberNames = newOnes + '_' + repeatedGroups.astype(str)
                newOnes = np.where(repeatedGroups, numberNames, newOnes) 
                dataFrame['Particulars'] = newOnes
                dataFrame['Particulars'] = pd.Categorical(dataFrame['Particulars'], ordered=True) 
                dataFrame.columns.name = title
                finalThings.append(dataFrame)
                print('=============================DATAFRAME MADE=================================')
            else:
                print('==============================datalist if empty==============================')

    return finalThings                    

In [88]:
balanceSheet1 = financial_statements(path)
print(len(balanceSheet1))
balanceSheet1[0].head(40)

SHIVAM CEMENTS LIMITED
==============================datalist if empty==============================
Financial Position
['16th', 'July', '2025', '15th', 'July', '2024']
16th
July
2025
15th
July
2024
==============thCounter============= 2
(Ashad 32, 2082) (Ashad 31, 2081)
Assets
Non Current Assets
Property, Plant & Equipment 3,885,988,681 3,613,074,969
[('Property, Plant & Equipment', '3,885,988,681', '3,613,074,969')]
Intangible Assets 11,303,875 3,396,234
[('Intangible Assets', '11,303,875', '3,396,234')]
Investment in Quoted Shares 131,208,000 119,280,000
[('Investment in Quoted Shares', '131,208,000', '119,280,000')]
Investment in Subsidiaries 4,755,208,800 4,585,208,800
[('Investment in Subsidiaries', '4,755,208,800', '4,585,208,800')]
Other Non-Current Assets 37,123,251 34,464,296
[('Other Non-Current Assets', '37,123,251', '34,464,296')]
Total Non Current Assets 8,820,832,606 8,355,424,299
[('Total Non Current Assets', '8,820,832,606', '8,355,424,299')]
Current Assets
Inventories

Financial Position,section,Particulars,16th July 2025,15th July 2024
0,Non Current Assets,"Property, Plant & Equipment","3,885,988,681","3,613,074,969"
1,Non Current Assets,Intangible Assets,"11,303,875","3,396,234"
2,Non Current Assets,Investment in Quoted Shares,"131,208,000","119,280,000"
3,Non Current Assets,Investment in Subsidiaries,"4,755,208,800","4,585,208,800"
4,Non Current Assets,Other Non-Current Assets,"37,123,251","34,464,296"
5,Non Current Assets,Total Non Current Assets,"8,820,832,606","8,355,424,299"
6,Current Assets,Inventories,"1,735,417,180","1,874,856,439"
7,Financial Assets,Trade Receivables,"819,967,247","1,126,644,809"
8,Financial Assets,Cash & Cash Equivalent,"651,816,544","240,488,471"
9,Financial Assets,Bank Balance (Other than Cash & Cash Equivalent),"400,000,000","200,000,000"


In [16]:
balanceSheet2 = financial_statements(file1)[0]
balanceSheet2.head(40)

Financial Position,section,Particulars,17th Oct 2025,16th Oct 2024
0,Non Current Assets,"Property, Plant & Equipment","3,895,234,883","3,580,529,913"
1,Non Current Assets,Intangible Assets,"9,682,252","173,569"
2,Non Current Assets,Investment in Quoted Shares,"120,353,520","119,280,000"
3,Non Current Assets,Investment in Subsidiaries,"4,755,208,800","4,585,208,800"
4,Non Current Assets,Other Non-Current Assets,"36,523,251","37,520,160"
5,Non Current Assets,Total Non Current Assets,"8,817,002,705","8,322,712,442"
6,Current Assets,Inventories,"1,778,825,479","1,759,821,823"
7,Financial Assets,Trade Receivables,"974,268,125","1,183,443,478"
8,Financial Assets,Cash & Cash Equivalent,"267,362,547","184,825,856"
9,Financial Assets,Bank Balance (Other than Cash & Cash Equivalent),"650,000,000",-


In [11]:
balanceSheet2

Profit & Loss,section,Particulars,Upto 1 Qtr 2082-83,Upto 1 Qtr 2081-82
0,Income,Revenue From Operations,"1,533,309,617","1,406,380,579"
1,Income,Cost of Sales,"1,206,737,196","1,302,730,722"
2,Income,Gross Profit,"326,572,421","103,649,857"
3,Income,Other Income,"103,656,894","105,743,699"
4,Expenses,Administration Expenses,"61,879,452","48,041,806"
5,Expenses,Selling and Distribution Expenses,"131,702,941","138,594,633"
6,Expenses,Finance Cost,"3,212,745","2,762,198"
7,Expenses,Total Expenses,"196,795,139","189,398,636"
8,Expenses,Profit/(Loss) Before Tax,"233,434,176","19,994,920"
9,Expenses,Current Tax,"16,439,092","859,882"


In [17]:
pd.merge(balanceSheet1, balanceSheet2, on='Particulars', how='outer').sort_values('Particulars')

Financial Position,section_x,Particulars,16th July 2025,15th July 2024,section_y,17th Oct 2025,16th Oct 2024
0,Financial Assets,Bank Balance (Other than Cash & Cash Equivalent),"400,000,000","200,000,000",Financial Assets,"650,000,000",-
1,Financial Assets,Cash & Cash Equivalent,"651,816,544","240,488,471",Financial Assets,"267,362,547","184,825,856"
2,Financial Liabilities,Deferred Tax Liabilities,"200,342,221","158,457,390",Financial Liabilities,"209,405,154","148,253,763"
3,Equity,Equity Share Capital,"5,456,808,500","5,027,000,000",Equity,"5,456,808,500","5,027,000,000"
4,Financial Liabilities,Financial Liabilities_Other Financial Liabilities,"12,160,661","3,432,662",Financial Liabilities,"12,160,661",-
5,Financial Liabilities,Financial Liabilities_Other Financial Liabilit...,"113,050,483","48,481,990",Financial Liabilities,"96,261,536","29,669,374"
6,Financial Assets,Income Tax Assets (Net),"6,198,709","15,396,894",Financial Assets,-,"15,232,046"
7,NaN,Income Tax Liabilities (Net),NaN,NaN,Financial Liabilities,"9,955,221",-
8,Non Current Assets,Intangible Assets,"11,303,875","3,396,234",Non Current Assets,"9,682,252","173,569"
9,Current Assets,Inventories,"1,735,417,180","1,874,856,439",Current Assets,"1,778,825,479","1,759,821,823"


In [33]:
from pathlib import Path

folderPath = 'D:/shivamData'
files = [x for x in Path(folderPath).iterdir() if x.is_file()]
combinedData = []

for file in files:
    tempList = financial_statements(file)
    print('\n\ncurrent file:', file)
    display(tempList[0])
    combinedData.append(tempList)

combinedBalanceSheet = []
for ind, tables in enumerate(combinedData):
    balanceSheet = tables[0]
    if ind == 0:
        combinedBalanceSheet = balanceSheet
        continue
    combinedBalanceSheet = pd.merge(combinedBalanceSheet, balanceSheet, how='outer', on='Particulars')
    display(combinedBalanceSheet.head(40))
    print(combinedBalanceSheet.columns)

combinedBalanceSheet



current file: D:\shivamData\040d3-1st-quarter-financial-statement_fy-81-082.pdf


Financial Position,section,Particulars,"16th Oct 2024 (Ashoj 30, 2081)","17th Oct 2023 (Ashoj 30, 2080)"
0,Non-Current Assets,"Property, Plant & Equipment","3,580,529,913","3,128,084,485"
1,Non-Current Assets,Intangible Assets,"173,569","346,792"
2,Non-Current Assets,Investment in Quoted Shares,"119,280,000","128,822,400"
3,Non-Current Assets,Investment in Subsidiaries,"4,585,208,800","4,262,135,800"
4,Non-Current Assets,Other Non-Current Assets,"37,520,160","55,895,160"
5,Non-Current Assets,Total Non-Current Assets,"8,322,712,442","7,575,284,637"
6,Current Assets,Inventories,"1,759,821,823","1,879,253,130"
7,Financial Assets,Trade Receivables,"1,183,443,478","1,687,609,586"
8,Financial Assets,Cash & Cash Equivalent,"184,825,856","157,887,088"
9,Financial Assets,Others Financial Assets,"11,619,724","28,306,501"




current file: D:\shivamData\3e778-3rd-quarter-financial-statement_fy079-80.pdf


Financial Position,section,Particulars,12th Apr 2024 14th,Jan 2024 13th Apr 2023
0,"Provisions 11,222,560 9,971,139 4,293,625",Income Tax Liabilities (Net) -,"6,588,117","13,666,900"




current file: D:\shivamData\52407-2nd-quarter-financial-statement_fy-81-082.pdf


...

Financial Position,section,Particulars,14th Jan 2024,14th Jan 2023
0,Non-Current Assets,"Property, Plant & Equipment","3,606,040,403","3,633,904,350"
1,Non-Current Assets,Intangible Assets,"253,397","487,686"
2,Non-Current Assets,Investment in Quoted Shares,"121,665,600","168,542,640"
3,Non-Current Assets,Investment in Subsidiaries,"4,262,135,800","4,262,135,800"
4,Non-Current Assets,Other Non-Current Assets,"34,323,811","33,579,164"
5,Non-Current Assets,Total Non-Current Assets,"8,024,419,011","8,098,649,640"
6,Current Assets,Inventories,"1,700,696,213","1,823,136,191"
7,Financial Assets,Trade Receivables,"1,840,514,041","2,043,884,000"
8,Financial Assets,Cash & Cash Equivalent,"194,057,953","171,867,839"
9,Financial Assets,Others Financial Assets,"267,663,027","26,796,573"




current file: D:\shivamData\a06b5-4rd-quarter-financial-statement_fy-80-081.pdf


Financial Position,section,Particulars,15th July 2024,16th July 2023
0,Non-Current Assets,"Property, Plant & Equipment","3,606,676,545","3,607,472,395"
1,Non-Current Assets,Intangible Assets,"200,178","306,615"
2,Non-Current Assets,Investment in Quoted Shares,"119,280,000","134,786,400"
3,Non-Current Assets,Investment in Subsidiaries,"4,585,208,800","4,262,135,800"
4,Non-Current Assets,Other Non-Current Assets,"34,464,296","34,273,632"
5,Non-Current Assets,Total Non-Current Assets,"8,345,829,820","8,038,974,843"
6,Current Assets,Inventories,"1,871,476,408","1,687,795,030"
7,Financial Assets,Trade Receivables,"1,129,101,138","1,422,942,847"
8,Financial Assets,Cash & Cash Equivalent,"240,488,471","428,781,594"
9,Financial Assets,Others Financial Assets,"17,006,378","181,645,161"




current file: D:\shivamData\ced52-1th-quarter-financial-statement_fy-080-81.pdf


IndexError: list index out of range

In [89]:
thisFile = financial_statements('D:/shivamData/ced52-1th-quarter-financial-statement_fy-080-81.pdf')
thisFile[0].head(45)

SHIVAM CEMENTS LIMITED
==============================datalist if empty==============================
Financial Position
['Shivam', 'Cements']
Shivam
Cements
==============thCounter============= 2
17th Oct 2023 16th July 2023 17th Oct, 2022
(Ashoj 30, 2080) (Ashad 31, 2080) (Ashoj 31, 2079)
Assets
Non-Current Assets
Property, Plant & Equipment 3,128,084,485 3,249,133,622 3,527,241,703
Intangible Assets 346,792 378,692 595,496
Investment in Quoted Shares 219,851,575 219,851,575 212,571,575
Investment in Subsidiaries 4,262,135,800 4,262,135,800 4,262,135,800
Other Non-Current Assets 55,895,160 34,273,632 35,917,121
Total Non-Current Assets 7,666,313,812 7,765,773,321 8,038,461,694
Current Assets
Inventories 1,879,253,130 1,695,411,083 2,289,412,735
Financial Assets
Trade Receivables 1,687,609,586 1,437,548,798 1,735,991,698
Cash & Cash Equivalent 157,887,088 428,781,594 128,025,858
Others Financial Assets 28,306,501 112,553,904 27,146,321
Other Current Assets 504,940,679 172,288,986 147,7

IndexError: list index out of range